# Multi-Turn Evaluation with RAGAS & a Custom LLM Judge (Azure OpenAI)

A beginner-friendly tour of how to evaluate **multi-turn conversations** (a back-and-forth
chat, not a single question). We cover **two** complementary approaches:

| Part | Tool | What it gives you |
|---|---|---|
| **A** | **RAGAS** metrics | Pass/fail & numeric criteria, rubric scoring, goal accuracy, topic adherence |
| **B** | **Custom LLM judge** | A fully custom prompt that returns a structured JSON verdict |

For each tool you'll see **two ways to get data**:
- **From a JSON file** — load and score many conversations at once
- **Your own samples** — an editable cell you can copy/paste conversations into

**Prerequisites**
- Python 3.10+ (Google Colab works out of the box)
- An Azure OpenAI resource: endpoint, API key, deployment name, API version
- The two sample datasets shipped alongside this notebook:
  `ragas_multiturn_banking_support_dataset_1.json` and `ragas_multiturn_banking_support_dataset_2.json`
  (on Colab, upload them to the working directory first)

> Run the cells top to bottom.


## 1. Install dependencies

Run once per session. **On Colab, restart the runtime after installing** before continuing.


In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain-openai==0.3.28 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.2 \
    python-dotenv==1.1.1 \
    openpyxl==3.1.5
print('Install complete. If running on Colab, restart the runtime before proceeding.')

## 2. Configure Azure OpenAI credentials

All three approaches use the same Azure OpenAI model as the judge, so we set the
credentials **once** here and reuse them everywhere.

> ⚠️ **Security note:** the API key is hard-coded for convenience. Prefer environment
> variables / Colab secrets, and **rotate any key that has been shared**.


In [ ]:
import os
# from dotenv import load_dotenv
# load_dotenv()  # optional: load these from a local .env file instead

# Shared Azure OpenAI credentials (used by RAGAS and the custom judge)
AZURE_API_KEY     = "YOUR_AZURE_API_KEY"
AZURE_ENDPOINT    = "YOUR_AZURE_ENDPOINT"
AZURE_API_VERSION = "YOUR_AZURE_API_VERSION"  # GPT-5 requires a 2025+ API version
AZURE_DEPLOYMENT  = "YOUR_AZURE_DEPLOYMENT"       # name of your GPT-5 deployment

os.environ["AZURE_OPENAI_API_KEY"]     = AZURE_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"]    = AZURE_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_API_VERSION

print("Azure credentials set. Deployment:", AZURE_DEPLOYMENT)

### RAGAS evaluator LLM

Wrap Azure OpenAI so RAGAS can use it as the judge. (Any `DeprecationWarning` about
the wrapper is safe to ignore on RAGAS 0.3.x.)


In [ ]:
from ragas.llms.base import LangchainLLMWrapper
from langchain_openai import AzureChatOpenAI

# GPT-5 rejects any temperature other than 1. RAGAS otherwise forces a low
# temperature on the judge LLM (0.01, or 0.3 when n > 1). This subclass pins
# temperature=1 for BOTH generate_text (sync) and agenerate_text (async), so the
# notebook works regardless of which RAGAS code path a metric happens to use.
class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

# RAGAS evaluator LLM -- the "judge" used by all RAGAS metrics below.
evaluator_llm = GPT5LangchainLLMWrapper(
    AzureChatOpenAI(
        deployment_name=AZURE_DEPLOYMENT,
        temperature=1,  # GPT-5 only accepts the default temperature (1)
        azure_endpoint=AZURE_ENDPOINT,
        openai_api_version=AZURE_API_VERSION,
        openai_api_key=AZURE_API_KEY,
    ),
    bypass_temperature=True,
)
print("RAGAS evaluator LLM ready (GPT-5 compatible, temperature=1).")

## 3. The multi-turn data format

A multi-turn example is an ordered list of **turns** that alternate between the user and
the assistant.

- **RAGAS** uses `MultiTurnSample(user_input=[...])` built from `HumanMessage` and
  `AIMessage` objects.

Our JSON files use this schema (one entry per conversation):

```json
{
  "dataset": [
    {
      "history": [ {"role": "user", "content": "..."},
                   {"role": "assistant", "content": "..."} ],
      "question": "the latest user message",
      "api_response_details": { "search_summarization": "the assistant answer to evaluate" },
      "ground_truth": "the ideal answer (reference)"
    }
  ]
}
```


# Part A - RAGAS Multi-Turn Evaluation

The flow below starts with the normal user path: upload an Excel file and run the eligible metrics. After that, the notebook shows the bundled JSON examples, then one or two hand-written user examples.

Metric used throughout this section:
- `AspectCritic`: asks the judge LLM to return a binary pass/fail score for a plain-English rule.


## 4. Upload an Excel file and run eligible RAGAS metrics

Upload the workbook that contains your multi-turn outputs. The cell reads `case_id`, `turn`, `question`, `expected_answer`, `actual_answer`, and the `ragas` column. When `ragas` is present, it is used as the conversation source; otherwise the cell builds the conversation from `question` and `actual_answer`. Metrics that need references are run only when `expected_answer` is available.

Metrics used in this Excel category:
- `AspectCritic`: checks whether the assistant responses stay grounded and avoid unsupported facts.
- `SimpleCriteriaScore`: gives a 0-5 helpfulness score for completeness, clarity, and usefulness.
- `RubricsScore`: gives a 1-5 overall quality score using explicit rubric descriptions.
- `AgentGoalAccuracyWithoutReference`: checks whether the assistant appears to satisfy the user goal without an expected answer.
- `AgentGoalAccuracyWithReference`: checks goal completion against `expected_answer` when every conversation has a reference.


In [ ]:
import ast
from pathlib import Path

import pandas as pd
from IPython.display import display

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, MultiTurnSample
from ragas.messages import HumanMessage, AIMessage
from ragas.metrics import (
    AspectCritic,
    SimpleCriteriaScore,
    RubricsScore,
    AgentGoalAccuracyWithoutReference,
    AgentGoalAccuracyWithReference,
)

try:
    from google.colab import files
    uploaded = files.upload()
    excel_path = next(iter(uploaded))
except Exception:
    excel_path = input("Path to the uploaded Excel file: ").strip().strip('"')

if not excel_path:
    raise ValueError("No Excel file was provided.")

raw_df = pd.read_excel(excel_path)
raw_df.columns = [str(col).strip().lower().replace(" ", "_") for col in raw_df.columns]

column_aliases = {
    "input": "question",
    "user_input": "question",
    "prompt": "question",
    "actual_output": "actual_answer",
    "generated_output": "actual_answer",
    "output": "actual_answer",
    "expected_output": "expected_answer",
    "expected": "expected_answer",
    "ground_truth": "expected_answer",
}
rename_map = {
    source: target
    for source, target in column_aliases.items()
    if source in raw_df.columns and target not in raw_df.columns
}
raw_df = raw_df.rename(columns=rename_map)

required_any = {"ragas", "question", "actual_answer"}
if not required_any.intersection(raw_df.columns):
    raise ValueError(
        "The workbook must contain a 'ragas' column, or at least 'question' and 'actual_answer' columns."
    )

if "case_id" not in raw_df.columns:
    raw_df["case_id"] = [f"ROW-{idx:04d}" for idx in range(len(raw_df))]
if "turn" in raw_df.columns:
    raw_df["turn_sort"] = pd.to_numeric(raw_df["turn"], errors="coerce")
else:
    raw_df["turn_sort"] = raw_df.groupby("case_id").cumcount() + 1

raw_df = raw_df.sort_values(["case_id", "turn_sort"], kind="stable")

def _text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

def parse_ragas_messages(value):
    """Parse a spreadsheet ragas cell like [HumanMessage(content='...'), AIMessage(content='...')]."""
    text = _text(value)
    if not text:
        return []

    tree = ast.parse(text, mode="eval")
    if not isinstance(tree.body, (ast.List, ast.Tuple)):
        raise ValueError("The ragas cell must be a list of HumanMessage(...) and AIMessage(...).")

    messages = []
    for node in tree.body.elts:
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            raise ValueError("Each ragas entry must be HumanMessage(...) or AIMessage(...).")
        if node.func.id not in {"HumanMessage", "AIMessage"}:
            raise ValueError(f"Unsupported ragas message type: {node.func.id}")

        kwargs = {kw.arg: kw.value for kw in node.keywords if kw.arg}
        content_node = kwargs.get("content")
        if not isinstance(content_node, ast.Constant) or not isinstance(content_node.value, str):
            raise ValueError("Each ragas message must have a string content=... value.")

        message_cls = HumanMessage if node.func.id == "HumanMessage" else AIMessage
        messages.append(message_cls(content=content_node.value))

    return messages

def build_reference(group):
    if "expected_answer" not in group.columns:
        return ""
    parts = []
    for _, row in group.iterrows():
        expected = _text(row.get("expected_answer"))
        question = _text(row.get("question"))
        if expected:
            prefix = f"Q: {question}\n" if question else ""
            parts.append(f"{prefix}Expected answer: {expected}")
    return "\n\n".join(parts)

def build_messages_from_rows(group):
    messages = []
    for _, row in group.iterrows():
        question = _text(row.get("question"))
        actual = _text(row.get("actual_answer"))
        if question:
            messages.append(HumanMessage(content=question))
        if actual:
            messages.append(AIMessage(content=actual))
    return messages

samples = []
sample_rows = []
for case_id, group in raw_df.groupby("case_id", sort=False):
    ragas_messages = []
    if "ragas" in group.columns:
        for _, row in group.iterrows():
            if _text(row.get("ragas")):
                ragas_messages = parse_ragas_messages(row.get("ragas"))
                break

    messages = ragas_messages or build_messages_from_rows(group)
    if not messages:
        continue

    reference = build_reference(group)
    sample_kwargs = {"user_input": messages}
    if reference:
        sample_kwargs["reference"] = reference

    samples.append(MultiTurnSample(**sample_kwargs))
    sample_rows.append(
        {
            "case_id": case_id,
            "turns": len(messages),
            "source": "ragas" if ragas_messages else "question_actual_answer",
            "has_reference": bool(reference),
        }
    )

if not samples:
    raise ValueError("No usable multi-turn conversations were found in the workbook.")

sample_summary = pd.DataFrame(sample_rows)
display(sample_summary)

metrics = []
metrics.append(
    AspectCritic(
        name="answer_groundedness_binary",
        definition=(
            "Return 1 if the assistant answers are grounded in the conversation and do not introduce unsupported facts. "
            "Otherwise return 0."
        ),
        llm=evaluator_llm,
    )
)
metrics.append(
    SimpleCriteriaScore(
        name="helpfulness_0_to_5",
        definition=(
            "Score from 0 to 5 how helpful, complete, and clear the assistant responses are for the user's requests "
            "(0 = unhelpful, 5 = fully resolves the conversation)."
        ),
        llm=evaluator_llm,
    )
)
metrics.append(
    RubricsScore(
        name="overall_quality_1_to_5",
        rubrics={
            "score1_description": "Incorrect, unsupported, or unhelpful.",
            "score2_description": "Partially helpful but contains major gaps or errors.",
            "score3_description": "Mostly helpful with some missing detail or clarity issues.",
            "score4_description": "Accurate and clear with only minor gaps.",
            "score5_description": "Fully accurate, clear, and complete for the whole conversation.",
        },
        llm=evaluator_llm,
    )
)
metrics.append(AgentGoalAccuracyWithoutReference(llm=evaluator_llm))

if sample_summary["has_reference"].all():
    metrics.append(AgentGoalAccuracyWithReference(llm=evaluator_llm))
else:
    print("Skipping AgentGoalAccuracyWithReference because at least one conversation has no expected_answer reference.")

print("Running metrics:", [getattr(metric, "name", metric.__class__.__name__) for metric in metrics])

result = evaluate(
    dataset=EvaluationDataset(samples=samples),
    metrics=metrics,
)

report_df = pd.concat([sample_summary.reset_index(drop=True), result.to_pandas().reset_index(drop=True)], axis=1)
display(report_df)

output_path = Path(excel_path).with_name(Path(excel_path).stem + "_ragas_report.xlsx")
report_df.to_excel(output_path, index=False)
print(f"Saved report to: {output_path}")


## 5. Load the conversations from JSON

We ship **two** datasets with the same schema. We use **Dataset 1** (`records`) below; **Dataset 2** (`records_1`) is an alternate you can swap in anywhere - they are interchangeable.

Metrics used in this JSON category:
- `AspectCritic`: checks custom binary rules such as groundedness and factual correctness.
- `SimpleCriteriaScore`: gives a numeric score for helpfulness or another criterion you define.
- `RubricsScore`: scores each conversation against a written 1-5 rubric.
- `AgentGoalAccuracyWithoutReference`: checks whether the assistant achieved the inferred user goal.
- `AgentGoalAccuracyWithReference`: checks goal completion against a reference answer or expected outcome.
- `TopicAdherenceScore`: checks whether the conversation stays within allowed topics.
- `ToolCallAccuracy` / `ToolCallF1`: evaluate expected tool/function calls when tool-call data is present.


In [ ]:
import json
from ragas.messages import HumanMessage, AIMessage
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset

with open("ragas_multiturn_banking_support_dataset_1.json", "r") as f:
    raw = json.load(f)

records = raw["dataset"]

In [ ]:
import json
from ragas.messages import HumanMessage, AIMessage
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset

with open("ragas_multiturn_banking_support_dataset_2.json", "r") as f:
    raw = json.load(f)

records_1 = raw["dataset"]

In [ ]:
print("Dataset 1 (records):  ", len(records), "conversations")
print("Dataset 2 (records_1):", len(records_1), "conversations")
print("\nKeys in a record:", list(records[0].keys()))
print("\nExample question:", records[0]["question"])
print("Example answer:  ", records[0]["api_response_details"]["search_summarization"])

## 6. Build samples from the JSON dataset

Three ways to turn JSON records into `MultiTurnSample`s, depending on what you want to score. The later JSON metric cells use the full-conversation samples from Approach 2 unless noted otherwise.


### Approach 1 — minimal (latest question + answer only)

Each record becomes a tiny 2-turn sample (just the final exchange).


In [ ]:
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage

samples = []

for item in records:
    q = item["question"]
    a = item["api_response_details"]["search_summarization"]

    sample = MultiTurnSample(
        user_input=[
            HumanMessage(content=q),
            AIMessage(content=a)
        ]
    )

    samples.append(sample)

print(samples)

### Approach 2 — full conversation per record (recommended)

Include each record's prior `history` **plus** the final exchange, so the judge sees the
complete multi-turn context. We'll evaluate these `json_samples` below.


In [ ]:
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage

# Convert one JSON record (history + final Q&A) into a full multi-turn sample.
def record_to_multiturn_sample(record):
    messages = []
    for turn in record.get("history", []):
        if turn["role"] == "user":
            messages.append(HumanMessage(content=turn["content"]))
        else:
            messages.append(AIMessage(content=turn["content"]))
    # add the final exchange we want to evaluate
    messages.append(HumanMessage(content=record["question"]))
    messages.append(AIMessage(content=record["api_response_details"]["search_summarization"]))
    return MultiTurnSample(user_input=messages)

json_samples = [record_to_multiturn_sample(r) for r in records]
print(f"Built {len(json_samples)} full multi-turn sample(s) from the JSON dataset.")

### Approach 3 — treat the whole log as one long conversation

Merges every record's turns into a single `MultiTurnSample` (useful for one continuous session).


In [ ]:
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage

# list of messages
all_messages = []

for item in records:
    q = item["question"]
    a = item["api_response_details"]["search_summarization"]

    all_messages.append(HumanMessage(content=q))
    all_messages.append(AIMessage(content=a))

# Now ONE sample contains ALL turns
sample = MultiTurnSample(
    user_input=all_messages
)

combined_samples = [sample]

print(combined_samples)

## 7. Evaluate the JSON conversations with an Aspect Critic

Metric used: `AspectCritic` - returns 1 when the answer stays grounded in the banking facts from the conversation and 0 when it hallucinates or goes unsupported.


In [ ]:
from ragas.metrics import AspectCritic

definition = """
Return 1 if the AI answer stays within the banking facts and account information from the conversation and does not hallucinate.
Otherwise return 0.
"""

aspect_critic = AspectCritic(
    name="banking_groundedness_check",
    definition=definition,
    llm=evaluator_llm
)

In [ ]:
from ragas.dataset_schema import EvaluationDataset
from ragas import evaluate

result = evaluate(
    dataset=EvaluationDataset(samples=json_samples),
    metrics=[aspect_critic]
)

df = result.to_pandas()
print(df)

Save the scores to an Excel file you can download:

In [ ]:
df.to_excel("results.xlsx", index=False)

## 8. Another JSON criterion - answer correctness

Metric used: `AspectCritic` - this time the plain-English rule checks whether the assistant answer is factually correct for the customer question.


In [ ]:
definition = """
Return 1 if the AI answer is factually correct for the customer's banking question.
Return 0 if the answer is partially incorrect or fully incorrect.
"""

correctness_metric = AspectCritic(
    name="banking_answer_correctness",
    definition=definition,
    llm=evaluator_llm
)

In [ ]:
from ragas.dataset_schema import EvaluationDataset
from ragas import evaluate

result = evaluate(
    dataset=EvaluationDataset(samples=json_samples),
    metrics=[correctness_metric]
)

df = result.to_pandas()
print(df)

In [ ]:
df.to_excel("results_1.xlsx", index=False)

## 9. More built-in RAGAS multi-turn metrics for JSON samples

`AspectCritic` gives a yes/no answer. RAGAS ships several other metrics that also work on multi-turn conversations - they all reuse the same `json_samples` we built in Section 6 (or your own). Here are the most useful ones for a beginner.

Metrics demonstrated below:
- `SimpleCriteriaScore`: scores a custom criterion on a numeric scale.
- `RubricsScore`: chooses a score using your written rubric descriptions.
- `AgentGoalAccuracyWithoutReference`: evaluates goal completion from the conversation alone.
- `AgentGoalAccuracyWithReference`: evaluates goal completion against a reference outcome.
- `TopicAdherenceScore`: measures whether the assistant stays on approved topics.
- `ToolCallAccuracy` / `ToolCallF1`: evaluate tool-call behavior when the sample includes tool calls and reference tool calls.


### SimpleCriteriaScore — a numeric score

Like `AspectCritic`, but returns a **number on a scale you define** instead of 0/1.


In [ ]:
from ragas.metrics import SimpleCriteriaScore
from ragas.dataset_schema import EvaluationDataset
from ragas import evaluate

helpfulness = SimpleCriteriaScore(
    name="helpfulness_0_to_5",
    definition="Score from 0 to 5 how helpful and complete the assistant's answers are for the customer's banking request (0 = unhelpful, 5 = fully resolves it).",
    llm=evaluator_llm,
)

result = evaluate(
    dataset=EvaluationDataset(samples=json_samples),
    metrics=[helpfulness],
)
result.to_pandas()

### RubricsScore — score against a rubric

You spell out what each score (1–5) means; the judge picks the best fit. Edit the
descriptions to match your own quality bar.


In [ ]:
from ragas.metrics import RubricsScore

rubrics = {
    "score1_description": "Incorrect or unhelpful; ignores the customer's request.",
    "score2_description": "Partially helpful but has major errors or missing information.",
    "score3_description": "Mostly helpful but lacks clarity or some detail.",
    "score4_description": "Accurate and clear, with only minor gaps.",
    "score5_description": "Fully and accurately resolves the customer's request.",
}

rubric_score = RubricsScore(name="banking_rubric_1_to_5", rubrics=rubrics, llm=evaluator_llm)

result = evaluate(
    dataset=EvaluationDataset(samples=json_samples),
    metrics=[rubric_score],
)
result.to_pandas()

### AgentGoalAccuracy — did the assistant achieve the user's goal?

`AgentGoalAccuracyWithoutReference` infers the user's goal from the conversation and checks
whether it was met. `AgentGoalAccuracyWithReference` compares the outcome against an
expected result (here we reuse each record's `ground_truth`).


In [ ]:
from ragas.metrics import (
    AgentGoalAccuracyWithoutReference,
    AgentGoalAccuracyWithReference,
)
from ragas.dataset_schema import MultiTurnSample

# 1) Without a reference outcome
goal_no_ref = AgentGoalAccuracyWithoutReference(llm=evaluator_llm)
print("AgentGoalAccuracy (without reference):")
print(evaluate(dataset=EvaluationDataset(samples=json_samples),
               metrics=[goal_no_ref]).to_pandas())

# 2) With a reference outcome (the expected result for each conversation)
samples_with_goal = [
    MultiTurnSample(user_input=s.user_input, reference=rec["ground_truth"])
    for rec, s in zip(records, json_samples)
]
goal_with_ref = AgentGoalAccuracyWithReference(llm=evaluator_llm)
print("\nAgentGoalAccuracy (with reference):")
print(evaluate(dataset=EvaluationDataset(samples=samples_with_goal),
               metrics=[goal_with_ref]).to_pandas())

### TopicAdherenceScore — did it stay on allowed topics?

Useful to catch off-domain answers. Each sample needs a list of `reference_topics`
(the topics the assistant is allowed to discuss).


In [ ]:
from ragas.metrics import TopicAdherenceScore

allowed_topics = [
    "account balances and transactions",
    "cards and payments",
    "loans and interest rates",
    "disputes and fraud",
    "general banking support",
]

# attach the allowed topics to each conversation
topic_samples = [
    MultiTurnSample(user_input=s.user_input, reference_topics=allowed_topics)
    for s in json_samples
]

topic_adherence = TopicAdherenceScore(llm=evaluator_llm, mode="precision")

result = evaluate(
    dataset=EvaluationDataset(samples=topic_samples),
    metrics=[topic_adherence],
)
result.to_pandas()

### Tool-using agents — ToolCallAccuracy & ToolCallF1

For agents that call **tools/functions**, RAGAS can check whether the assistant made the
*expected* tool calls. These need each sample to include `AIMessage(tool_calls=[...])` and
a `reference_tool_calls` list — our text-only banking data has none, so here is the shape
rather than a run:

```python
from ragas.metrics import ToolCallAccuracy
from ragas.dataset_schema import MultiTurnSample
from ragas.messages import HumanMessage, AIMessage, ToolCall

sample = MultiTurnSample(
    user_input=[
        HumanMessage(content="Transfer $50 to my savings."),
        AIMessage(content="Sure.", tool_calls=[
            ToolCall(name="transfer_funds", args={"amount": 50, "to": "savings"})
        ]),
    ],
    reference_tool_calls=[
        ToolCall(name="transfer_funds", args={"amount": 50, "to": "savings"}),
    ],
)

result = evaluate(dataset=EvaluationDataset(samples=[sample]), metrics=[ToolCallAccuracy()])
```


## 10. Evaluate your own samples (one or two editable examples)

This is your **copy-paste area**: edit the two `MultiTurnSample` objects below with your own conversations. Each `HumanMessage` is a user turn and each `AIMessage` is an assistant turn.

Metric used in this user-example category:
- `AspectCritic`: checks the custom forgetfulness rule and returns 1 when the assistant completes all human requests without the user having to re-ask.


In [ ]:
from ragas.metrics import AspectCritic, SimpleCriteriaScore
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage, AIMessage
from ragas import evaluate


sample1 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="I need to increase my credit limit and check why my last transaction at Walmart was declined."
        ),
        AIMessage(
            content="Let's check the declined transaction first, the $234.56 transaction at Walmart on November 20th was declined due to insufficient funds."
        ),
        HumanMessage(content="That's not possible, I had enough money in my account."),
        AIMessage(
            content="I understand your concern. Let me check your account details... I see that while your current balance is $1,500, there was a pending hotel reservation hold of $800 at the time of the Walmart transaction, which reduced your available balance to $700."
        ),
        HumanMessage(content="Oh, I forgot about the hotel booking."),
        AIMessage(content="Is there anything else I can assist you with?"),
        HumanMessage(content="What about increasing my credit limit?"),
        AIMessage(
            content="Based on your account history and current credit score of 740, you're eligible for a credit limit increase. The maximum increase available is $2,000. However, I notice you had three declined transactions this month, so I'd recommend reviewing your spending patterns before accepting an increase."
        ),
    ]
)

sample2 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="I need to check why my last transaction at Target was declined."
        ),
        AIMessage(
            content="Let's look at the declined transaction. The $145.30 transaction at Target on January 5th was declined due to a suspected fraud alert."
        ),
        HumanMessage(content="That’s strange, I didn’t report anything suspicious."),
        AIMessage(
            content="I understand. Let me look deeper. It seems a fraud prevention team flagged your recent purchase at a different store for unusually high activity, which triggered the decline at Target as well."
        ),
        HumanMessage(content="Ah, that makes sense. I did shop a lot that day."),
        AIMessage(content="Is there anything else I can assist you with?"),
    ]
)

## 11. Define an Aspect Critic and evaluate the user examples

Metric used: `AspectCritic` - the criterion checks for **forgetfulness**, meaning whether the assistant handled every request without the user having to re-ask.


In [ ]:
definition = "Return 1 if the AI completes all Human requests fully without any rerequests; otherwise, return 0."

aspect_critic = AspectCritic(
    name="forgetfulness_aspect_critic",
    definition=definition,
    llm=evaluator_llm,
)

In [ ]:
result = evaluate(
    dataset=EvaluationDataset(samples=[sample1, sample2]),
    metrics=[aspect_critic],
)

result.to_pandas()

# Part B — Custom LLM-as-Judge (structured JSON output)

When the built-in metrics aren't enough, write your own judge prompt. Here the LLM reads
the full chat history, the latest question, and the answer, then returns a strict **JSON**
verdict (score, reason, highlights, categories, confidence).


## 12. The evaluation prompt

A detailed, banking-specific rubric. Edit the checklist to fit your own use case.


In [ ]:
evaluation_prompt = """
You are a specialist banking customer-support evaluation AI. Your task is to evaluate the FINAL MODEL ANSWER using the full multi-turn conversational context. You must check whether the answer is factually correct, non-hallucinated, safe, and relevant to both the latest question AND the preceding chat history. Use conservative, evidence-based judgment grounded in standard banking policies and consumer-finance regulations.

Inputs:
FULL CHAT HISTORY:
{history}

LATEST USER QUESTION:
{question}

MODEL ANSWER TO EVALUATE:
{answer}

Primary rule (binary scoring):
- Return 1 ONLY if the model answer contains zero incorrect factual claims, zero hallucinations, zero fabricated facts, AND remains consistent with all relevant details from the chat history.
- Return 0 if ANY incorrect statement, hallucination, misleading claim, omission of essential compliance or security info, or contradiction with earlier turns is found.

Multi-Turn Evaluation Checklist:
1. Accuracy — Are account details, fees, rates, and policy statements factually correct?
2. Hallucination — Does the answer add invented facts or fabricated details?
3. Context Consistency — Does it contradict or ignore earlier conversation turns?
4. Relevance — Does it correctly respond to the final user question using history when needed?
5. Safety — Could following this answer cause financial harm or violate compliance (e.g., unauthorized financial advice)?
6. Proper uncertainty — Does it avoid making up unavailable details?

Output requirements:
Return ONLY valid JSON (no text outside JSON):
{{
  "score": <0 or 1>,
  "reason": "<10–25 word explanation>",
  "highlights": ["<excerpt(s) from model answer that determined score>"],
  "categories": ["accuracy"|"hallucination"|"omission"|"safety"|"relevance"|"context"],
  "confidence": <0.0–1.0>
}}

Rules:
- ANY violation → score = 0.
- If uncertain info exists, model should explicitly say so rather than invent facts.
- Ensure strict JSON output without markdown formatting.

Now evaluate and return JSON only.
"""

## 13. Initialize the Azure OpenAI client

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

print("Azure GPT-5 client initialized.")

## 14. Run the judge over the conversations

For each record we inject its history, question, and answer into the prompt, then ask Azure OpenAI for structured JSON output.


In [ ]:
results = []

for idx, item in enumerate(records):

    # Multi-turn conversation history
    history_messages = item.get("history", [])

    # Convert list of messages into readable text block
    history_text = ""
    for turn in history_messages:
        history_text += f"{turn['role'].upper()}: {turn['content']}\n"

    question = item["question"]
    answer   = item["api_response_details"]["search_summarization"]

    # Inject into prompt
    prompt = evaluation_prompt.format(
        history=history_text,
        question=question,
        answer=answer
    )

    response = client.chat.completions.create(
        model=AZURE_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=1  # GPT-5 only accepts the default temperature (1)
    )

    raw_content = response.choices[0].message.content.strip()

    # Clean markdown fences
    if raw_content.startswith("```"):
        raw_content = raw_content.replace("```json", "").replace("```", "").strip()

    try:
        eval_json = json.loads(raw_content)
    except Exception as e:
        print(f"JSON parse error at record: {idx}")
        print("Raw output:", raw_content)
        continue

    results.append({
        "history": history_text,
        "question": question,
        "answer": answer,
        "score": eval_json.get("score"),
        "reason": eval_json.get("reason"),
        "highlights": eval_json.get("highlights"),
        "categories": eval_json.get("categories"),
        "confidence": eval_json.get("confidence")
    })

print("Evaluation completed.")

## 15. View the results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df

## Recap & next steps

You've now evaluated multi-turn conversations two ways:
- **RAGAS `AspectCritic`** — your own pass/fail criteria (forgetfulness, groundedness, correctness)
- **Custom LLM judge** — a structured JSON verdict you fully control

**Try next**
- Paste your own conversations into the editable cell (Section 5).
- Point the loaders in Section 4 at your own JSON (same schema).
- Add new `AspectCritic` definitions or new fields to the custom judge's JSON output.

📚 RAGAS docs: https://docs.ragas.io
